# EDA — Mendeley LBC Cervical Cancer
Analisi esplorativa del dataset: struttura, classi, dimensioni originali, qualità e visualizzazione esempi degradati.

In [ ]:
import sys
sys.path.insert(0, '..')
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
import pandas as pd
import torch
from collections import Counter

DATA_DIR = Path('../data/raw')
DEGRADED_DIR = Path('../data/degraded')
print('Setup completato.')

## 1. Struttura del dataset

In [ ]:
image_paths = list(DATA_DIR.rglob('*.png')) + list(DATA_DIR.rglob('*.jpg'))
print(f'Totale immagini trovate: {len(image_paths)}')

classes = {}
for p in image_paths:
    cls = p.parent.name
    classes[cls] = classes.get(cls, 0) + 1

print('\nDistribuzione per classe:')
for cls, count in sorted(classes.items(), key=lambda x: -x[1]):
    print(f'  {cls}: {count} immagini ({count/len(image_paths)*100:.1f}%)')

## 2. Distribuzione per classe (grafico)

In [ ]:
labels = list(classes.keys())
counts = list(classes.values())
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(labels)), counts, color=colors, edgecolor='white')
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels([l[:20]+'...' if len(l)>20 else l for l in labels], rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Numero immagini')
axes[0].set_title('Distribuzione per classe')
for i, v in enumerate(counts):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

axes[1].pie(counts, labels=[l[:25] for l in labels], colors=colors,
           autopct='%1.1f%%', startangle=140, textprops={'fontsize': 9})
axes[1].set_title('Proporzioni classi')

plt.tight_layout()
plt.savefig('../data/degraded/examples/eda_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Distribuzione dimensioni originali

In [ ]:
widths, heights = [], []
for p in image_paths:
    try:
        with Image.open(p) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except:
        pass

print(f'Larghezza  — min: {min(widths)}, max: {max(widths)}, media: {np.mean(widths):.0f}')
print(f'Altezza    — min: {min(heights)}, max: {max(heights)}, media: {np.mean(heights):.0f}')

size_counts = Counter(zip(widths, heights))
print('\nDimensioni più comuni:')
for size, count in size_counts.most_common(5):
    print(f'  {size[0]}x{size[1]}: {count} immagini')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=20, color='#4C72B0', edgecolor='white')
axes[0].set_xlabel('Larghezza (px)')
axes[0].set_ylabel('Frequenza')
axes[0].set_title('Distribuzione larghezze originali')
axes[0].axvline(256, color='red', linestyle='--', label='Target 256px')
axes[0].legend()

axes[1].hist(heights, bins=20, color='#DD8452', edgecolor='white')
axes[1].set_xlabel('Altezza (px)')
axes[1].set_ylabel('Frequenza')
axes[1].set_title('Distribuzione altezze originali')
axes[1].axvline(256, color='red', linestyle='--', label='Target 256px')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/degraded/examples/eda_sizes.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Campioni per classe

In [ ]:
by_class = {}
for p in image_paths:
    cls = p.parent.name
    by_class.setdefault(cls, []).append(p)

n_classes = len(by_class)
n_cols = 4
fig, axes = plt.subplots(n_classes, n_cols, figsize=(n_cols * 3, n_classes * 3))

for row, (cls, paths) in enumerate(sorted(by_class.items())):
    for col in range(n_cols):
        ax = axes[row][col] if n_classes > 1 else axes[col]
        if col < len(paths):
            img = Image.open(paths[col]).convert('RGB').resize((256, 256))
            ax.imshow(img)
            if col == 0:
                ax.set_ylabel(cls[:25], fontsize=8, rotation=0, ha='right', va='center')
        ax.axis('off')

plt.suptitle('Campioni per classe (256x256)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../data/degraded/examples/eda_samples_per_class.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Ground truth vs immagini degradate

In [ ]:
noise_levels = [0.005, 0.01, 0.05, 0.1]
gt_dir = DEGRADED_DIR / 'test' / 'gt'

if not gt_dir.exists():
    print('Esegui prima: python scripts/preprocess.py')
else:
    gt_files = sorted(gt_dir.glob('*.pt'))[:4]
    n_rows = len(gt_files)
    n_cols = len(noise_levels) + 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))

    def to_img(t):
        return (t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

    for row, gt_path in enumerate(gt_files):
        gt = torch.load(gt_path)
        axes[row][0].imshow(to_img(gt))
        axes[row][0].set_title('Ground Truth' if row == 0 else '')
        axes[row][0].axis('off')
        for col, nl in enumerate(noise_levels):
            deg_path = DEGRADED_DIR / 'test' / f'noise_{nl}' / gt_path.name
            deg = torch.load(deg_path)
            axes[row][col + 1].imshow(to_img(deg))
            axes[row][col + 1].set_title(f'noise={nl}' if row == 0 else '')
            axes[row][col + 1].axis('off')

    plt.suptitle('Ground Truth vs Degraded (blur σ=2 + noise)', fontsize=12)
    plt.tight_layout()
    plt.savefig('../data/degraded/examples/eda_gt_vs_degraded.png', dpi=120, bbox_inches='tight')
    plt.show()

## 6. Statistiche pixel (media e std per canale)

In [ ]:
import torchvision.transforms as T

transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])
sample_paths = image_paths[:200]

means, stds = [], []
for p in sample_paths:
    try:
        img = transform(Image.open(p).convert('RGB'))
        means.append(img.mean(dim=[1, 2]).numpy())
        stds.append(img.std(dim=[1, 2]).numpy())
    except:
        pass

means = np.array(means)
stds = np.array(stds)
channels = ['R', 'G', 'B']
print('Statistiche pixel (campione 200 immagini):')
for i, ch in enumerate(channels):
    print(f'  {ch} — mean: {means[:, i].mean():.3f}, std: {stds[:, i].mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors_ch = ['#e74c3c', '#2ecc71', '#3498db']
for i, (ch, color) in enumerate(zip(channels, colors_ch)):
    axes[i].hist(means[:, i], bins=30, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribuzione media canale {ch}')
    axes[i].set_xlabel('Valore medio pixel [0,1]')
    axes[i].set_ylabel('Frequenza')
plt.tight_layout()
plt.savefig('../data/degraded/examples/eda_pixel_stats.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Riepilogo dataset

In [ ]:
print('=' * 50)
print('RIEPILOGO DATASET')
print('=' * 50)
print(f'Totale immagini:      {len(image_paths)}')
print(f'Numero classi:        {len(classes)}')
print(f'Dimensione target:    256x256')
print(f'Normalizzazione:      mean=0.5, std=0.5 → [-1, 1]')
print(f'Blur:                 Gaussian σ=2, kernel=9')
print(f'Noise levels:         {noise_levels}')
print(f'Split:                70% train / 15% val / 15% test')
print(f'Seed:                 42')
print('=' * 50)